In [53]:
print('hello world')

hello world


In [54]:
import docx
import json

In [55]:
doc = docx.Document('F:/ai_rag_based_chatbot/customer-support-rag-chat-system/resource/documents/global_shipping_policy.docx')

In [56]:
def extract_tag_content(line, tag):

    opening = f"<{tag}>"
    closing = f"</{tag}>"

    return (
        line.replace(opening, "")
            .replace(closing, "")
            .strip()
    )

In [57]:
lines = [para.text for para in doc.paragraphs]
lines[:4]

['<t> Global Shipping Policy </t>',
 '<h1>Overview</h1>',
 '<p>This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce.</p>',
 '<h1>Eligibility</h1>']

In [58]:
nodes = []
inside_paragraph = False
paragraph_buffer = []

for line in lines:

    line = line.strip()
    # blank line
    if not line:
        continue
    # actual line
    if inside_paragraph:

        if line.endswith("</p>"):
            
            clean = line.replace("</p>","")
            if clean:
                paragraph_buffer.append(clean)
            text = "\n".join(paragraph_buffer)
            nodes.append({
                        "type": "paragraph",
                        "text": text
                        })
            inside_paragraph = False
            paragraph_buffer = []
            
        else:
            paragraph_buffer.append(line)
            continue

            


    if line.startswith("<t>"):

      document_name = extract_tag_content(line, "t")
      nodes.append({
            "type": "title",
            "text": extract_tag_content(line, "t")
        })
            

    elif line.startswith("<h1>"):

        nodes.append({
            "type": "heading",
            "level": 1,
            "text": extract_tag_content(line, "h1")
        })

    elif line.startswith("<h2>"):
        nodes.append({
            "type" : "subheading",
            "level": 2,
            "text" : extract_tag_content(line, "h2")
        })

    elif line.startswith("<p>"):
        inside_paragraph = True
        

        if line.endswith("</p>"):
            
            inside_paragraph = False
            nodes.append({
                "type": "paragraph",
                "text": extract_tag_content(line, "p")
            })
        else:
            clean = line.replace("<p>","").strip()
            if clean:
                paragraph_buffer.append(clean)
        
        continue


    elif line.startswith("<li>"):

        nodes.append({
            "type": "list_item",
            "text": extract_tag_content(line, "li")
        })

    elif line.startswith("<faq-q>"):

        nodes.append({
            "type": "faq_question",
            "text": extract_tag_content(line, "faq-q")
        })

    elif line.startswith("<faq-a>"):

        nodes.append({
            "type": "faq_answer",
            "text": extract_tag_content(line, "faq-a")
        })

json_doc = {'document_name': document_name,
            'nodes': nodes}

json_output = json.dumps(json_doc, indent=2)
    

    

In [59]:
print(json_output)

{
  "document_name": "Global Shipping Policy",
  "nodes": [
    {
      "type": "title",
      "text": "Global Shipping Policy"
    },
    {
      "type": "heading",
      "level": 1,
      "text": "Overview"
    },
    {
      "type": "paragraph",
      "text": "This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce."
    },
    {
      "type": "heading",
      "level": 1,
      "text": "Eligibility"
    },
    {
      "type": "list_item",
      "text": "Applies to all products sold on the platform."
    },
    {
      "type": "list_item",
      "text": "Shipping availability depends on customer location."
    },
    {
      "type": "list_item",
      "text": "Certain regions may have restricted delivery."
    },
    {
      "type": "list_item",
      "text": "International shipping availability varies by product category."
    },
    {
    

In [60]:
# Raw Document
#     ↓
# Parser
#     ↓
# Normalized Semantic Nodes (IR)
#     ↓
# Enrichment Layer
#     ↓
# Chunking
#     ↓
# Embeddings
#     ↓
# Vector DB

Stage 3
Enrichment Layer

NOW you enrich nodes with:

global metadata
local metadata
hierarchy
relationships

Example:

{
  "type": "paragraph",
  "text": "Express shipping fee: 299 rs.",
  
  "global_metadata": {
    "document_type": "shipping_policy",
    "department": "logistics",
    "version": 1
  },

  "local_metadata": {
    "section": "Shipping Fees",
    "parent_heading": "Shipping Fees",
    "subsection": null
  }
}

THIS is where metadata belongs.

NOT before parsing.

Because parser first discovers structure.

Why Metadata AFTER Parsing?

Because parser identifies:

headings
hierarchy
sections
relationships

Without parsing:

you don't know local context
you don't know semantic boundaries

In [61]:
# Enrichment Layer

print(json_doc['nodes'][0])

{'type': 'title', 'text': 'Global Shipping Policy'}


In [62]:
class Buffer:
    def __init__(self):

        self.heading = None
        self.subheading = None
        self.title = None

    def get_path(self):
        
        path = []
        if self.title:
            path.append(self.title)
        
        if self.heading:
            path.append(self.heading)

        if self.subheading:
            path.append(self.subheading)
        
        return path
    
    def get_metadata(self):
        local_metadata = {
                            "title":self.title,
                            "heading":self.heading,
                            "subheading":self.subheading 
                            }
        return local_metadata
    
class EnrichNode:
    node_id = 0

    def __init__(self, _type, text, local_metadata, path):
        
        self._type = _type
        self.text = text
        self.local_metadata = local_metadata
        self.path = path
        EnrichNode.node_id +=1

    def get_node(self):
        return {
                    "node_id"       : EnrichNode.node_id,
                    "type"          : self._type,
                    "text"          : self.text,
                    "local_metadata": self.local_metadata,
                    "path"          : self.path
                }        


buffer = Buffer()
enrich_nodes = []
SEMANTIC_TYPES = [
    "paragraph",
    "list_item",
    "faq_question",
    "faq_answer"
]


for node in json_doc['nodes']:

    if node['type'] == 'title':
        
        buffer.title = node['text']
        

    elif node['type'] == 'heading':
        
        buffer.heading = node['text']
        buffer.subheading = None

    elif node['type'] == 'subheading':

        buffer.subheading = node['text']
        


    elif node['type'] in SEMANTIC_TYPES:

    
        enrich_node = EnrichNode(_type=node['type'],
                                 text=node['text'],
                                 local_metadata=buffer.get_metadata(),
                                 path=buffer.get_path())
        

        enrich_nodes.append(enrich_node.get_node())    
        

# create enrich state representation

enrich_doc = {"nodes": enrich_nodes}
    
print(json.dumps(enrich_doc,indent=2))    

{
  "nodes": [
    {
      "node_id": 1,
      "type": "paragraph",
      "text": "This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce.",
      "local_metadata": {
        "title": "Global Shipping Policy",
        "heading": "Overview",
        "subheading": null
      },
      "path": [
        "Global Shipping Policy",
        "Overview"
      ]
    },
    {
      "node_id": 2,
      "type": "list_item",
      "text": "Applies to all products sold on the platform.",
      "local_metadata": {
        "title": "Global Shipping Policy",
        "heading": "Eligibility",
        "subheading": null
      },
      "path": [
        "Global Shipping Policy",
        "Eligibility"
      ]
    },
    {
      "node_id": 3,
      "type": "list_item",
      "text": "Shipping availability depends on customer location.",
      "local_metadata": {
   

In [71]:
# grouping node by semantic location

groups = {}
#FAQ_TYPES = ["faq_question","faq_answer"]

for node in enrich_doc['nodes']:

    # if node['type'] in FAQ_TYPES:
    #     continue


    key = str(node['path'])

    if key not in groups:
        groups[key] = []
        
    groups[key].append(node) 

In [72]:
for k, v in groups.items():
    print(k," -> ",len(v),v)

['Global Shipping Policy', 'Overview']  ->  1 [{'node_id': 1, 'type': 'paragraph', 'text': 'This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce.', 'local_metadata': {'title': 'Global Shipping Policy', 'heading': 'Overview', 'subheading': None}, 'path': ['Global Shipping Policy', 'Overview']}]
['Global Shipping Policy', 'Eligibility']  ->  4 [{'node_id': 2, 'type': 'list_item', 'text': 'Applies to all products sold on the platform.', 'local_metadata': {'title': 'Global Shipping Policy', 'heading': 'Eligibility', 'subheading': None}, 'path': ['Global Shipping Policy', 'Eligibility']}, {'node_id': 3, 'type': 'list_item', 'text': 'Shipping availability depends on customer location.', 'local_metadata': {'title': 'Global Shipping Policy', 'heading': 'Eligibility', 'subheading': None}, 'path': ['Global Shipping Policy', 'Eligibility']}, {'node_id

In [67]:
class Chunk:
    chunk_id = 1

    def __init__(self,path,text):
        self.id = Chunk.chunk_id
        self.path = path
        self.text = text
        
        Chunk.chunk_id += 1

    def get_chunk(self):

        return {
                "chunk_id" : self.id,
                "path"     : self.path,
                "text"     : self.text
            }
        

In [ ]:
chunks = []
faq_question = None

for path,nodes in groups.items():
    
    buffer = [] # RESET BUFFER FOR NEW GROUP
    
    for node in nodes:

        if node['type'] == "faq-question":
            faq_question = node['text']

        elif node['type'] == "faq-answer":
            text = f"""
                    Q: {faq_question}

                    A: {node['text']}
                    """
            
                                

        buffer.append(node['text'])

    text = "\n".join(buffer)
    chunk = Chunk(path=path, text=text)
    chunks.append(chunk.get_chunk())


print(json.dumps(chunks,indent=2))
    

        
        

[
  {
    "chunk_id": 1,
    "path": "['Global Shipping Policy', 'Overview']",
    "text": "This policy defines shipping timelines, shipping fees, international shipping eligibility, shipment tracking procedures, and lost shipment handling for all customer orders placed through ABC Ecommerce."
  },
  {
    "chunk_id": 2,
    "path": "['Global Shipping Policy', 'Eligibility']",
    "text": "Applies to all products sold on the platform.\nShipping availability depends on customer location.\nCertain regions may have restricted delivery.\nInternational shipping availability varies by product category."
  },
  {
    "chunk_id": 3,
    "path": "['Global Shipping Policy', 'Delivery Timelines', 'Domestic Shipping']",
    "text": "Standard Shipping: 3-7 business days\nExpress Shipping: 1-3 business days"
  },
  {
    "chunk_id": 4,
    "path": "['Global Shipping Policy', 'Delivery Timelines', 'International Shipping']",
    "text": "Standard International: 7-15 business days\nPriority Internatio